In [1]:
import yaml
import shutil
from pathlib import Path
import sys
sys.path.append('../')
from src.data.utils import create_dataset_manifest

In [2]:
try:
    with open("../config/params.yml", "r") as f:
        params = yaml.safe_load(f)
except FileNotFoundError:
    print("ERROR: config/params.yml no encontrado.")
    params = None
except Exception as e:
    print(f"ERROR al leer config/params.yml: {e}")
    params = None

if params:
    seed = params.get('random_seed', 42)

    exp_params = params.get('experiment', {})
    exp_name = exp_params.get('name', 'default_experiment')
    exp_base_dir = Path(exp_params.get('base_dir', '../experiments'))
    experiment_dir = exp_base_dir / exp_name

    # Directorio del exp y guardar config
    experiment_dir.mkdir(parents=True, exist_ok=True)
    config_run_path = experiment_dir / "config_run.yml"

    try:
        shutil.copyfile("../config/params.yml", config_run_path)
        print(f"Directorio del experimento creado en: {experiment_dir.resolve()}")
        print(f"Configuración de esta ejecución guardada en: {config_run_path.resolve()}")
    except Exception as e:
        print(f"ERROR al copiar params.yml: {e}")

    # Definir subdurectorios de salida del experimento
    dp_params = params.get('data_pipeline', {})
    processed_subdir = experiment_dir / dp_params.get('splitted_clips_output_subdir', 'processed_data/clips_splitted')
    manifests_subdir = experiment_dir / dp_params.get('manifests_subdir', 'results/tables')

    # Crear subdirectorios de resultados
    manifests_subdir.mkdir(parents=True, exist_ok=True)

    # Definir fuentes de datos
    ds_params = params.get('data_source', {})
    raw_videos_dir = Path(ds_params.get('raw_videos_dir', '../data/raw/dataset_videos_original'))
    input_clips_dir = Path(ds_params.get('input_clips_dir', '../data/interim/clips_original'))

    # parámetros de pre procesamiento
    vp_params = params.get('video_processing', {})
    clip_length = vp_params.get('clip_length', 16)
    max_segments = vp_params.get('max_segments_per_video', 32)
    overlapping = vp_params.get('overlapping', True)
    stride = vp_params.get('stride', 8)
    balance_mode = dp_params.get('balance_mode', 'none')
    max_ratio = vp_params.get('balance_max_ratio', 1.2)
    split_ratios = vp_params.get('split_ratios', [0.64, 0.16, 0.20])
    train_r, val_r, test_r = split_ratios
    
else:
    print("No se pudo cargar la configuración.")

Directorio del experimento creado en: D:\Dataset\experiments\exp_13
Configuración de esta ejecución guardada en: D:\Dataset\experiments\exp_13\config_run.yml


## 1. Extracción de Clips

In [3]:
if not params:
    print("No se ha cargado la configuración, cargar primero el archivo params.yml.")
    raise ValueError("No se ha cargado la configuración.")

clips_output_path = input_clips_dir

#comando base
cmd_parts = [
    "../src/data/clip_sampler.py",
    f"--input_dir {raw_videos_dir}",
    f"--output_dir {clips_output_path}",
    f"--clip_length {clip_length}",
    f"--max_segments {max_segments}"
]

#--overlapping solo si es True
if overlapping:
    cmd_parts.append("--overlapping")
    cmd_parts.append(f"--stride {stride}")

#unir el comando
cmd = " ".join(cmd_parts)

%run {cmd}

Iniciando extracción de clips...
Modo de muestreo: Uniforme (Disperso)


Procesando 'Normal':  15%|█▌        | 29/192 [00:33<02:54,  1.07s/it]

Advertencia: Clip incompleto extraído de ..\data\raw\dataset_videos_recortados\Normal\Normal_Videos080_x264.mp4 comenzando en frame 1984. Se esperaban 16 frames, pero se obtuvieron 9.


Procesando 'Normal': 100%|██████████| 192/192 [03:53<00:00,  1.21s/it]

Extracción de clips completada.


## 2. Generación de manifest CSV 

In [4]:
base_manifest_csv = manifests_subdir / f"manifest_{input_clips_dir.name}.csv"

#clips_dir = "../data/interim/1_clips_original"
#unbalanced_manifest_csv = "../results/tables/1_manifest_clips_original.csv"

try:
    create_dataset_manifest(input_dir=str(input_clips_dir), output_csv=str(base_manifest_csv))
    print(f"Manifiesto generado en: {base_manifest_csv.resolve()}")

    manifest_for_next_step = base_manifest_csv
    clips_dir_for_next_step = input_clips_dir
except Exception as e:
    print(f"ERROR al generar el manifiesto: {e}")
    manifest_for_next_step = None
    clips_dir_for_next_step = None

Escaneando directorio: ..\data\interim\clips_recortados_13
Archivo CSV creado con 318 registros en: ..\experiments\exp_13\results\tables\manifest_clips_recortados_13.csv
  Clase 'Normal': 192 videos
  Clase 'Robbery': 126 videos
Manifiesto generado en: D:\Dataset\experiments\exp_13\results\tables\manifest_clips_recortados_13.csv


## 3. Balancear el dataset (opcional)

In [5]:
print(f"Balanceo del dataset (Modo: {balance_mode})")

if balance_mode != 'none' and manifest_for_next_step:
    # Los clips balanceados van en data/interim (raíz del proyecto)
    # El manifest va en el directorio del experimento
    clips_dir_name = input_clips_dir.name  # Solo 'clips_original_05'
    
    # Directorio de clips balanceados en data/interim (raíz del proyecto)
    project_root = Path("..").resolve()  # Desde notebooks/ subir a la raíz
    balanced_clips_dir = project_root / "data" / "interim" / f"{clips_dir_name}_{balance_mode}"
    
    # Manifest en el directorio del experimento
    balanced_manifest_csv = manifests_subdir / f"manifest_{clips_dir_name}_{balance_mode}.csv"

    print(f"Directorio balanceado: {balanced_clips_dir.resolve()}")
    print(f"Manifest balanceado: {balanced_manifest_csv.resolve()}")

    try:
        %run ../src/data/balancer.py \
            --input_csv {manifest_for_next_step} \
            --output_dir {balanced_clips_dir} \
            --output_csv {balanced_manifest_csv} \
            --mode {balance_mode} \
            --max_ratio {max_ratio} \
            --seed {seed}

        manifest_for_next_step = balanced_manifest_csv
        clips_dir_for_next_step = balanced_clips_dir
        print(f"Balanceo completado. Clips guardados en: {balanced_clips_dir.resolve()}")
    except Exception as e:
        print(f"ERROR al balancear el dataset: {e}")
        manifest_for_next_step = None
        clips_dir_for_next_step = None
elif not manifest_for_next_step:
    print("Omitiendo balanceo del dataset debido a error en el paso anterior.")
else:
    print("Omitiendo balanceo del dataset.")

if manifest_for_next_step:
    print(f"\nManifest a usar para división: {manifest_for_next_step.name}")
    print(f"Directorio de clips a usar para división: {clips_dir_for_next_step.resolve()}\n")

Balanceo del dataset (Modo: none)
Omitiendo balanceo del dataset.

Manifest a usar para división: manifest_clips_recortados_13.csv
Directorio de clips a usar para división: D:\Dataset\data\interim\clips_recortados_13



## 4. Dividir el dataset en splits (train/val/test)

In [6]:
if manifest_for_next_step and clips_dir_for_next_step:

    splitted_clips_dir = processed_subdir #"../data/processed/splits/dataset_clips_splitted"
    splitted_manifest_csv = manifests_subdir / "manifest_splitted.csv" #"../results/tables/manifest_clips_splitted.csv"

    print(f"usando manifest: {manifest_for_next_step.name}")
    print(f"Leyendo clips de: {clips_dir_for_next_step.resolve()}")
    print(f"Guardando splits en: {splitted_clips_dir.resolve()}")

    try:
        #train_ratio = 0.64
        #val_ratio = 0.16
        #test_ratio = 0.20

        %run ../src/data/splitter.py \
            --input_csv {manifest_for_next_step} \
            --output_dir {splitted_clips_dir} \
            --output_csv {splitted_manifest_csv} \
            --ratios {train_r} {val_r} {test_r} \
            --seed {seed}

        print(f"\nPreparación de datos completada para '{exp_name}'.")
        print(f"Dataset final se encuentra en: {splitted_clips_dir.resolve()}")
    except Exception as e:
        print(f"ERROR al dividir el dataset: {e}")
else:
    print("Omitiendo división del dataset debido a error en pasos anteriores.")

usando manifest: manifest_clips_recortados_13.csv
Leyendo clips de: D:\Dataset\data\interim\clips_recortados_13
Guardando splits en: D:\Dataset\experiments\exp_13\processed_data\clips_splitted

Distribución de videos por split y clase:
class  Normal  Robbery
split                 
test       39       25
train     122       81
val        31       20

Copiando directorios de clips a D:\Dataset\experiments\exp_13\processed_data\clips_splitted


Copiando videos: 100%|██████████| 318/318 [00:08<00:00, 36.81it/s]


Nuevo manifiesto con splits guardado en: D:\Dataset\experiments\exp_13\results\tables\manifest_splitted.csv

Preparación de datos completada para 'exp_13'.
Dataset final se encuentra en: D:\Dataset\experiments\exp_13\processed_data\clips_splitted
